In [3]:
import xarray as xr
import numpy as np
import functions
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib as mpl
import seaborn as sns
import cmcrameri as cm
from scipy import stats
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.markers import MarkerStyle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import statsmodels.api as sm
import pymannkendall as mk

In [2]:
rpath = '/nird/datapeak/NS9600K/astridbg/arctic-cld-feedbacks/data/observations_reanalysis_data/'
start_year = 2003
end_year = 2025
Arctic_limit = 59
spring_summer_months = [3, 4, 5, 6, 7, 8]
spring_summer_label = 'MAMJJA'
summer_months = [6, 7, 8]

In [76]:
def calculate_spatial_trends(data, start_year, end_year):
    n_lat = len(data.lat)
    n_lon = len(data.lon)
    data = data.sel(year=slice(start_year, end_year))

    trends = np.ones((n_lat, n_lon))*np.nan
    p_values = np.ones((n_lat, n_lon))*np.nan
    for ilon in range(n_lon):
        for ilat in range(n_lat):
            y = np.array(data.isel(lon=ilon, lat=ilat).values)
            # Check for NaNs
            if np.isnan(y).any():
                slope = np.nan
                p = np.nan
            # Check if all values are the same
            elif len(set(y)) == 1:
                slope = 0
                p = 0
            else:
                slope, _, _, _, p = functions.theilslopes_mk_prewhitened_test(y)

            trends[ilat, ilon] = slope
            p_values[ilat, ilon] = p
    
    ds = xr.Dataset(data_vars=dict(trends=(['lat', 'lon'], trends), p_values=(['lat', 'lon'], p_values)), 
                    coords = dict(lat = data.lat, lon = data.lon))

    print(ds)
    
    return ds

## Extract data and select Arctic region

### Snow cover data

Url: https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C00756

Robinson, David A.; Estilow, Thomas W.; and NOAA CDR Program (2012): NOAA Climate Data Record (CDR) of Northern Hemisphere (NH) Snow Cover Extent (SCE), Version 1. NOAA National Centers for Environmental Information. doi:10.7289/V5N014G9 [20.02.2026].

In [4]:
filepath = rpath+'snow_cover/nhsce_v01r01_19661004_20260202.nc'
sce = xr.open_dataset(filepath)

# Select times with meaningful data
sce = sce.sel(time=slice('1975-01-01','2026-01-01'))

# Select area
land_weights = sce.land # Extract land weights
latitude_weights = sce.latitude >= Arctic_limit # Select latitudes
 # Create weights that include all selected land areas
snow_cover_extent = sce['snow_cover_extent']*land_weights*latitude_weights

#Convert from fraction to percentage
snow_cover_extent = snow_cover_extent*100

### Soil moisture data

In [5]:
SM_ERA5 = xr.open_dataset(rpath+'soil_moisture/SM_ERA5_1970_2025_10cm_20cm.nc')

# Convert units from m3/m3 to kg/m2
SM_ERA5_converted = SM_ERA5['swvl_10cm']*100 # Conversion factor = 1000 kg/m3 * 0.10 m

### Cloud cover data

In [6]:
filepath = rpath+'cloud_cover/CERES_EBAF-TOA_Ed4.2.1_Subset_200003-202512.nc'
CERES  = xr.open_dataset(filepath)

# Calculate area-weighted regional average using land mask
cloud_cover = CERES['cldarea_total_daynight_mon'].sel(lat=slice(Arctic_limit, 90))

### Temperature data

In [7]:
filepath = rpath + 'temperature/ERA5_t2m_global.nc'
temp_ERA5 = xr.open_dataset(filepath)

# Re-index
temp_ERA5 = temp_ERA5.rename({'latitude':'lat', 'longitude':'lon', 'valid_time':'time'})
temp_ERA5 = temp_ERA5.reindex(lat=list(reversed(temp_ERA5.lat)))
lons = np.array(temp_ERA5.coords['lon'])
lons[np.where(lons<0)] = 360 + lons[np.where(lons<0)]
temp_ERA5.coords['lon'] = lons
temp_ERA5 = temp_ERA5.sortby(temp_ERA5.lon)

temp_ERA5 = temp_ERA5['t2m'].sel(lat=slice(Arctic_limit, 90))

## Calculate annual averages for different seasons

In [ ]:
# Snow cover extent

sce_season = snow_cover_extent.sel(time = snow_cover_extent['time.month'].isin(spring_summer_months)) # Select April, May, June, July and August
sce_season = sce_season.groupby(sce_season['time.year']).mean('time')

# Soil moisture ERA5-Land

SM_ERA5_season = SM_ERA5_converted.sel(time = SM_ERA5_converted['time.month'].isin(summer_months)) # Select June, July and August
SM_ERA5_season = SM_ERA5_season.groupby(SM_ERA5_season['time.year']).mean('time')


# Cloud cover

cc_season = cloud_cover.sel(time = cloud_cover['time.month'].isin(summer_months)) # Select June, July and August
cc_season = cc_season.groupby(cc_season['time.year']).mean('time')

# Temperature

temp_season = temp_ERA5.sel(time = temp_ERA5['time.month'].isin(summer_months)) # Select June, July and August
temp_season = temp_season.groupby(temp_season['time.year']).mean('time')


In [ ]:
cc_trends = calculate_spatial_trends(cc_season, start_year=2003, end_year=2025)
cc_trends.to_netcdf(rpath+'cloud_cover/CERES_trends_summer_'+str(start_year)+'_'+str(end_year)+'.nc')

<xarray.Dataset> Size: 180kB
Dimensions:   (lat: 31, lon: 360)
Coordinates:
  * lat       (lat) float32 124B 59.5 60.5 61.5 62.5 ... 86.5 87.5 88.5 89.5
  * lon       (lon) float32 1kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
Data variables:
    trends    (lat, lon) float64 89kB -0.08174 -0.08174 ... 0.1692 0.1692
    p_values  (lat, lon) float64 89kB 0.4282 0.4282 0.7917 ... 0.5612 0.5612


In [ ]:
SM_trends = calculate_spatial_trends(SM_ERA5_season, start_year=2003, end_year=2025)
SM_trends.to_netcdf(rpath+'soil_moisture/SM_ERA5_trends_summer_'+str(start_year)+'_'+str(end_year)+'.nc')